# Sanity check: β → 0 limit

Claim: at β=0 the Gibbs state is maximally mixed, so the optimizer's angles should land near zero.

Note: the circuit isn't literally the identity at zero angles — a fixed `H`+`CX` Bell-pair prefix always runs first. The real question is whether *zero correction* on top of that prefix is already correct.

We use `BETA_ZERO = 1e-8` instead of exact 0 (`beta**(-varsigma)` is singular there — same workaround `experiment.py` already uses). We reimplement `verify_exact_cost.py`'s noiseless cost inline, since `cost_function.py` was rewritten to require a real/simulated backend.

In [ ]:
import sys
sys.path.insert(0, '.')  # run from this folder (gibbs_tfd_reproduction/)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, state_fidelity, Pauli
from scipy.optimize import minimize

from tfd_circuit import TFDCircuit
from ground_truth import GibbsState, TFDState, default_subsystem_hamiltonian

VARSIGMA = 1.57
BETA_ZERO = 1e-8


def exact_cost(params, beta, varsigma=VARSIGMA):
    # mirrors verify_exact_cost.py, inlined to avoid a backend dependency
    g1, g2, a1, a2 = params
    sv = TFDCircuit(g1, g2, a1, a2).get_state_vector()
    exp_X_A = sv.expectation_value(Pauli("IIXI")).real + sv.expectation_value(Pauli("IIIX")).real
    exp_X_B = sv.expectation_value(Pauli("XIII")).real + sv.expectation_value(Pauli("IXII")).real
    exp_ZZ_A = sv.expectation_value(Pauli("IIZZ")).real
    exp_ZZ_B = sv.expectation_value(Pauli("ZZII")).real
    exp_XX_BA = sv.expectation_value(Pauli("XIXI")).real + sv.expectation_value(Pauli("IXIX")).real
    exp_ZZ_BA = sv.expectation_value(Pauli("ZIZI")).real + sv.expectation_value(Pauli("IZIZ")).real
    return (exp_X_A + exp_X_B + varsigma * (exp_ZZ_A + exp_ZZ_B)
            - beta ** (-varsigma) * (exp_XX_BA + exp_ZZ_BA))


def wrap(theta, period):
    # maps theta into (-period/2, period/2] -- collapses angles that are
    # equivalent under the real periodicity (pi for gamma1/gamma2, 2*pi
    # for alpha1/alpha2 -- see section 4) onto one canonical value
    return ((theta + period / 2) % period) - period / 2

## 1. Ground truth self-check

Gibbs state should be ≈ ¼·I at β→0, and match the exact TFD state's partial trace.

In [ ]:
H_AA = default_subsystem_hamiltonian()
gibbs_zero = GibbsState(H_AA).gibbs_density_matrix(beta=BETA_ZERO)

print("Gibbs state at beta -> 0 (expect ~0.25 * I):")
print(np.round(gibbs_zero, 4))

tfd_sv_zero = Statevector(TFDState(H_AA, H_AA).tfd_state_vector(beta=BETA_ZERO))
tfd_traceout_zero = partial_trace(tfd_sv_zero, [2, 3])
fid_ground_truth = state_fidelity(DensityMatrix(gibbs_zero), tfd_traceout_zero)
print(f"\nfidelity(Gibbs, exact TFD partial trace) at beta->0: {fid_ground_truth:.10f}")
assert fid_ground_truth > 1 - 1e-6

## 2. Zero angles, no optimization

Does the fixed Bell-pair prefix alone already give the right state?

In [ ]:
sv_zero_angles = TFDCircuit(0, 0, 0, 0).get_state_vector()
rho_A_zero_angles = partial_trace(sv_zero_angles, [2, 3])
fid_zero_angles = state_fidelity(DensityMatrix(gibbs_zero), rho_A_zero_angles)

print(f"fidelity(rho_A at angles=0, true Gibbs(beta->0)): {fid_zero_angles:.10f}")
print(f"cost at angles=0: {exact_cost([0, 0, 0, 0], BETA_ZERO):.4f}")
assert fid_zero_angles > 1 - 1e-6

## 3. Optimizer, 25 random nonzero starts

Same COBYLA optimizer the app uses, run 25 times from independent random nonzero initial guesses, to see whether it reliably finds a state with the right fidelity regardless of where it starts — not just from one particular starting point.

In [ ]:
N_TRIALS = 25
rng = np.random.default_rng(42)

rows = []
for i in range(N_TRIALS):
    # nonzero random start: each component in [-pi,-0.1] U [0.1,pi]
    magnitude = rng.uniform(0.1, np.pi, size=4)
    sign = rng.choice([-1, 1], size=4)
    x0 = magnitude * sign

    result = minimize(exact_cost, x0=x0, args=(BETA_ZERO,), method='COBYLA',
                       options={'maxiter': 300, 'rhobeg': 0.5})

    sv_opt = TFDCircuit(*result.x).get_state_vector()
    rho_A_opt = partial_trace(sv_opt, [2, 3])
    fid = state_fidelity(DensityMatrix(gibbs_zero), rho_A_opt)

    # gamma1/gamma2's real period is pi, alpha1/alpha2's is 2*pi (section 4) --
    # wrap into a canonical range so e.g. -3.14 and 0 show up as the same value
    g1, g2 = wrap(result.x[0], np.pi), wrap(result.x[1], np.pi)
    a1, a2 = wrap(result.x[2], 2 * np.pi), wrap(result.x[3], 2 * np.pi)

    rows.append({
        "trial": i,
        "g1_0": x0[0], "g2_0": x0[1], "a1_0": x0[2], "a2_0": x0[3],
        "g1": g1, "g2": g2, "a1": a1, "a2": a2,
        "fidelity": fid,
    })

results_df = pd.DataFrame(rows).set_index("trial").round(4)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(results_df.index, results_df["fidelity"], color="tab:blue")
ax.axhline(1.0, color="tab:green", linestyle="--", linewidth=1, label="fidelity = 1")
ax.set_xlabel("trial (random init)")
ax.set_ylabel("fidelity to true rho_A")
lo = results_df["fidelity"].min()
ax.set_ylim(lo - 0.0005, 1.0005)
ax.set_title(f"Fidelity across {N_TRIALS} random initializations at beta->0")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

print(f"fidelity: min={results_df['fidelity'].min():.6f}  "
      f"mean={results_df['fidelity'].mean():.6f}  max={results_df['fidelity'].max():.6f}")
print(f"trials with fidelity < 1-1e-3: {(results_df['fidelity'] < 1 - 1e-3).sum()} / {N_TRIALS}")
assert (results_df["fidelity"] > 1 - 1e-3).all()

Fidelity stays ≈1 in every one of the 25 trials, and now that `g1`/`g2` are shown wrapped modulo π (their real period — see section 4), they cluster tightly near 0 instead of splitting between ~0 and ~±π. `a1`/`a2` are wrapped modulo 2π just to keep them in a tidy range — they still scatter freely, as expected from the flat direction section 4 proves.

## Extra check: does swapping α₁ ↔ α₂ matter?

γ's aren't tested here — their cost genuinely depends on the angle (∝ −cos², minimized at 0 and ±π), so swapping γ₁↔γ₂ values wouldn't show anything new. α₁, α₂ are different: `RXX(α₁)` then `RZZ(α₂)` both act on an already-Bell-paired pair of qubits, and a Bell state is a simultaneous `XX`/`ZZ` eigenstate — so each rotation only ever contributes a global phase, in either order. Pick nonzero, unequal (a, b) and compare (α₁=a, α₂=b) against (α₁=b, α₂=a).

In [ ]:
def fid_at(g1, g2, a1, a2):
    sv = TFDCircuit(g1, g2, a1, a2).get_state_vector()
    rho_A = partial_trace(sv, [2, 3])
    return state_fidelity(DensityMatrix(gibbs_zero), rho_A)

rng2 = np.random.default_rng(7)
print(f"{'a':>8} {'b':>8} {'fid(a,b)':>14} {'fid(b,a)':>14} {'cost(a,b)':>16} {'cost(b,a)':>16}")
for _ in range(6):
    a, b = rng2.uniform(-3, 3, 2)
    fid_ab, fid_ba = fid_at(0, 0, a, b), fid_at(0, 0, b, a)
    cost_ab = exact_cost([0, 0, a, b], BETA_ZERO)
    cost_ba = exact_cost([0, 0, b, a], BETA_ZERO)
    print(f"{a:8.3f} {b:8.3f} {fid_ab:14.10f} {fid_ba:14.10f} {cost_ab:16.2f} {cost_ba:16.2f}")
    assert np.isclose(fid_ab, fid_ba, atol=1e-9)
    assert np.isclose(fid_ab, 1.0, atol=1e-9)

print("\nall swap pairs: fidelity unaffected by swapping alpha1 <-> alpha2, both exactly 1.0")

## Conclusion

- Ground truth checks out at β→0 (Gibbs ≈ ¼·I, matches exact TFD partial trace).
- **Zero angles alone already give the exact right state** — fidelity 1.0000000000.
- Across 25 random nonzero starts, the optimizer **always** lands on fidelity ≥ 0.999 — the prepared state reliably converges to the correct β→0 result.
- γ₁, γ₂ genuinely matter for the cost (∝ −cos²), with two tied-best points at 0 and ±π — that's why they land on either before wrapping, not a bug. α₁, α₂ are an exact flat direction (global phase only) — any value is equally optimal, for both fidelity and cost, in either order (swap check above). So the precise claim is about the **state converging correctly**, not literally all four angles going to zero.

## Animated trajectory: γ₁, γ₂ converging during optimization

Section 3 showed this numerically — 25 *finished* runs, wrapped γ's clustering near 0. This animates what happens *during* a run instead: COBYLA's `callback` hands us (γ₁, γ₂) after every iteration, for several random starts at once, so you can watch them move. Axes span [−π, π] so both known-optimal points (0 and ±π — see section 4) are visible if a trajectory heads for the edge instead of the center.

In [ ]:
N_ANIM_TRIALS = 50
anim_rng = np.random.default_rng(123)
trajectories = []

for _ in range(N_ANIM_TRIALS):
    magnitude = anim_rng.uniform(0.1, np.pi, size=4)
    sign = anim_rng.choice([-1, 1], size=4)
    x0 = magnitude * sign

    path = [(x0[0], x0[1])]  # frame 0 = the starting point itself

    def callback(xk, path=path):  # default arg binds *this* path, not the loop var
        path.append((xk[0], xk[1]))

    minimize(exact_cost, x0=x0, args=(BETA_ZERO,), method='COBYLA',
             callback=callback, options={'maxiter': 300, 'rhobeg': 0.5})
    trajectories.append(np.array(path))

print("trajectory lengths (iterations recorded):", [len(t) for t in trajectories])

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML
from pathlib import Path

max_len = max(len(t) for t in trajectories)
# tab10 only has 10 distinct colors and would repeat/alias at N=50 --
# hsv cycles the full color wheel, giving each trajectory a distinct hue.
colors = plt.cm.hsv(np.linspace(0, 1, N_ANIM_TRIALS, endpoint=False))

# A bit of margin beyond [-pi, pi] so the +-pi targets sit comfortably
# inside the frame instead of being clipped right at the border.
MARGIN = 0.5

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(-np.pi - MARGIN, np.pi + MARGIN)
ax.set_ylim(-np.pi - MARGIN, np.pi + MARGIN)
ax.set_xlabel("gamma1")
ax.set_ylabel("gamma2")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)

# Both 0 and +-pi are equally valid per axis (section 4's -cos^2 argument),
# so every (gamma1, gamma2) combination of {-pi, 0, pi} is a correct target
# -- not just the center. Mark all nine; a trajectory heading for an edge
# or corner is arriving somewhere right, not drifting off.
TARGET_VALUES = (-np.pi, 0, np.pi)
for tx in TARGET_VALUES:
    for ty in TARGET_VALUES:
        ax.plot(tx, ty, marker="*", color="gold", markeredgecolor="black",
                 markeredgewidth=0.8, markersize=16, zorder=5)
ax.plot([], [], marker="*", color="gold", markeredgecolor="black",
         markeredgewidth=0.8, markersize=16, linestyle="None",
         label="targets: gamma in {-pi, 0, pi}")
ax.legend(loc="upper right", fontsize=8)

# Thinner/more transparent than the N=6 version -- 50 overlapping full-opacity
# lines would just turn into visual noise.
lines = [ax.plot([], [], '-', color=colors[i], alpha=0.35, linewidth=1.0)[0] for i in range(N_ANIM_TRIALS)]
dots = [ax.plot([], [], 'o', color=colors[i], markersize=5, alpha=0.85)[0] for i in range(N_ANIM_TRIALS)]


def update(frame):
    for i, traj in enumerate(trajectories):
        idx = min(frame, len(traj) - 1)
        lines[i].set_data(traj[:idx + 1, 0], traj[:idx + 1, 1])
        dots[i].set_data([traj[idx, 0]], [traj[idx, 1]])
    ax.set_title(f"gamma1, gamma2 during optimization  (step {frame})  --  {N_ANIM_TRIALS} trials")
    return lines + dots


anim = FuncAnimation(fig, update, frames=max_len, interval=120, blit=False)

GIF_PATH = Path("plots") / "gamma_trajectory.gif"
GIF_PATH.parent.mkdir(exist_ok=True)
anim.save(str(GIF_PATH), writer=PillowWriter(fps=8))
plt.close(fig)
print(f"saved {GIF_PATH}")

HTML(anim.to_jshtml())